# TrialMatch-LLM: full GPU pipeline (Kaggle, 1x T4)
Runs M2-M5 end to end: baseline eval -> LoRA fine-tuning -> checkpoint selection on **val** -> one final **test** evaluation -> error analysis -> KV-cache / attention / speculative benchmarks -> figures -> adapter upload.

**Settings:** Accelerator = GPU T4 (x1 is enough), Internet = ON. Optional secret `HF_TOKEN` (write) to upload adapters.

In [ ]:
REPO_URL = 'https://github.com/Ali-Mazhar05/trialmatch-llm.git'   # <-- set
HF_ADAPTER_REPO = '<your-hf-user>/trialmatch-qwen2.5-1.5b-lora'    # <-- set (or None)
HF_DRAFT_REPO = '<your-hf-user>/trialmatch-qwen2.5-0.5b-draft-lora' # <-- set (or None)
TARGET = 'Qwen/Qwen2.5-1.5B-Instruct'
DRAFT = 'Qwen/Qwen2.5-0.5B-Instruct'

In [ ]:
!git clone -q $REPO_URL /kaggle/working/trialmatch-llm
%cd /kaggle/working/trialmatch-llm
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## M1 - data, split manifest, leakage checks

In [ ]:
!python scripts/prepare_data.py --tokenizer $TARGET
!python -m pytest -q tests/test_data_metrics.py

## M2 - baseline (before fine-tuning) on validation

In [ ]:
!python scripts/evaluate.py --model $TARGET --name base --split val

## M2 - LoRA fine-tuning (train split only)

In [ ]:
!python scripts/train_lora.py --model $TARGET --out runs/qwen15_lora --epochs 3 --lr 2e-4 --r 16 --alpha 32 --batch_size 4 --grad_accum 4 --load_dtype fp32 --amp fp16

## M4 - model selection on validation only

In [ ]:
!python scripts/select_best.py --run runs/qwen15_lora --model $TARGET
import json; BEST = json.load(open('runs/qwen15_lora/selection.json'))['best']['checkpoint']; print('BEST =', BEST)
!rm -rf runs/qwen15_lora/best_adapter && cp -r $BEST runs/qwen15_lora/best_adapter
!python scripts/evaluate.py --model $TARGET --name ft_best --split val --adapter runs/qwen15_lora/best_adapter

## M4 - FINAL held-out test evaluation (run once, same examples & decoding for base and fine-tuned)

In [ ]:
!python scripts/evaluate.py --model $TARGET --name base    --split test --final_test
!python scripts/evaluate.py --model $TARGET --name ft_best --split test --final_test --adapter runs/qwen15_lora/best_adapter
!python scripts/error_analysis.py --pred results/test/ft_best/predictions.jsonl
!python scripts/error_analysis.py --pred results/val/ft_best/predictions.jsonl

## M3 - correctness tests on GPU + KV-cache benchmark + attention benchmark

In [ ]:
!python -m pytest -q tests
!python scripts/bench_kv.py --model $TARGET --adapter runs/qwen15_lora/best_adapter --inputs 128,512,1024 --outputs 32,128,256 --repeats 3
!python scripts/bench_attention.py --repeats 5

## M5 - speculative decoding (base 0.5B draft), then a LoRA-tuned draft experiment

In [ ]:
!python scripts/bench_speculative.py --model $TARGET --adapter runs/qwen15_lora/best_adapter --draft $DRAFT --ks 1,2,4,6,8 --n_prompts 8 --repeats 2 --tag base_draft

In [ ]:
!python scripts/train_lora.py --model $DRAFT --out runs/qwen05_draft_lora --epochs 3 --lr 2e-4 --r 16 --alpha 32 --batch_size 4 --grad_accum 4 --load_dtype fp32 --amp fp16
!python scripts/bench_speculative.py --model $TARGET --adapter runs/qwen15_lora/best_adapter --draft $DRAFT --draft_adapter runs/qwen05_draft_lora/final_adapter --ks 1,2,4,6,8 --n_prompts 8 --repeats 2 --tag ft_draft

## Figures, tables, packaging

In [ ]:
!python scripts/make_figures.py --run runs/qwen15_lora --names base,ft_best
!cp runs/qwen15_lora/train_summary.json runs/qwen15_lora/run_config.json runs/qwen15_lora/selection.json results/ 2>/dev/null; mkdir -p results/draft && cp runs/qwen05_draft_lora/train_summary.json results/draft/ 2>/dev/null
!cd /kaggle/working && zip -qr trialmatch_outputs.zip trialmatch-llm/results trialmatch-llm/report/figures trialmatch-llm/report/generated_tables.tex trialmatch-llm/splits trialmatch-llm/runs/qwen15_lora/best_adapter trialmatch-llm/runs/qwen05_draft_lora/final_adapter && ls -lh trialmatch_outputs.zip

In [ ]:
# optional: upload adapters to the Hugging Face Hub (needs Kaggle secret HF_TOKEN)
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('HF_TOKEN')
    from huggingface_hub import HfApi
    api = HfApi(token=tok)
    for repo, folder in [(HF_ADAPTER_REPO, 'runs/qwen15_lora/best_adapter'), (HF_DRAFT_REPO, 'runs/qwen05_draft_lora/final_adapter')]:
        if repo and '<' not in repo:
            api.create_repo(repo, exist_ok=True)
            api.upload_folder(folder_path=folder, repo_id=repo)
            print('uploaded', repo)
except Exception as e:
    print('upload skipped:', e)